In [186]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
ROOT = Path.cwd().parent.parent
sys.path.append(str(ROOT))
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import seaborn as sns
import pickle

In [187]:
df_path = ROOT / 'data' /'master_data'/ '2016_to_2023_clustering_output_data.csv'
df = pd.read_csv(df_path)

In [188]:
cluster_cols = ['LCA_Class', 'k_proto_labs_25', 'hdb_labels_281_50']
df1 = df.groupby(['LCA_Class', 'hdb_labels_281_50']).size().reset_index(name='count')
df2 = df.groupby(['hdb_labels_281_50', 'k_proto_labs_25']).size().reset_index(name='count')

lca_unique = sorted(df['LCA_Class'].unique())
hdb_unique = sorted(df['hdb_labels_281_50'].unique())
k_proto_unique = sorted(df['k_proto_labs_25'].unique())

lca_mapping_dict = {k : v for v, k in enumerate(lca_unique)}
hdb_mapping_dict = {k : v + len(lca_unique) for v, k in enumerate(hdb_unique)}
k_proto_mapping_dict = {k : v + len(lca_unique) + len(hdb_unique) for v, k in enumerate(k_proto_unique)}

df1['LCA_Class'] = df1['LCA_Class'].map(lca_mapping_dict)
df1['hdb_labels_281_50'] = df1['hdb_labels_281_50'].map(hdb_mapping_dict)

df2['hdb_labels_281_50'] = df2['hdb_labels_281_50'].map(hdb_mapping_dict)
df2['k_proto_labs_25'] = df2['k_proto_labs_25'].map(k_proto_mapping_dict)

links_dict1 = df1.to_dict(orient='list')
links_dict2 = df2.to_dict(orient='list')

In [189]:
fig = go.Figure(data = [go.Sankey(
        node = dict(
            pad=15, 
            thickness = 20,
            line = dict(color='black', width = 0.5),
            label = lca_unique + hdb_unique + k_proto_unique,
            color = 'blue'
    ),

    link = dict(
        source = links_dict1['LCA_Class'] + links_dict2['hdb_labels_281_50'],
        target = links_dict1['hdb_labels_281_50'] + links_dict2['k_proto_labs_25'],
        value = links_dict1['count'] + links_dict2['count'],
    )
)]
)

In [190]:
fig

In [191]:
df = pd.read_csv(df_path)
ct = pd.crosstab(df['LCA_Class'], df['k_proto_labs_25'])
ct_pct = ct.div(ct.sum(axis=1), axis=0)  # row-normalized: % within each LCA_Class
threshold = 0.05
ct_pct_display = ct_pct.where(ct_pct >= threshold, np.nan)

fig = px.imshow(ct_pct_display, text_auto=".0%", aspect="auto", color_continuous_scale="Blues")
fig.update_layout(title="k_proto_labs_25 distribution within each LCA_Class", coloraxis_showscale = False)
fig.show()

In [ ]:
ari = adjusted_rand_score(df['LCA_Class'], df['k_proto_labs_25'])
nmi = normalized_mutual_info_score(df['LCA_Class'], df['k_proto_labs_25'])
print(f"ARI: {ari:.3f}")   # ~0 = no agreement, ~1 = identical clustering
print(f"NMI: {nmi:.3f}") 

ARI: 0.153
NMI: 0.376


In [193]:
df = pd.read_csv(df_path)
ct = pd.crosstab(df['LCA_Class'], df['hdb_labels_281_50'])
ct_pct = ct.div(ct.sum(axis=1), axis=0)  # row-normalized: % within each LCA_Class
threshold = 0.05
ct_pct_display = ct_pct.where(ct_pct >= threshold, np.nan)

fig = px.imshow(ct_pct_display, text_auto=".0%", aspect="auto", color_continuous_scale="Blues")
fig.update_layout(title="hdb_labels_281_50 distribution within each LCA_Class", coloraxis_showscale = False)
fig.show()

In [195]:
df = pd.read_csv(df_path)
ct = pd.crosstab(df['k_proto_labs_25'], df['hdb_labels_281_50'])
ct_pct = ct.div(ct.sum(axis=1), axis=0)  # row-normalized: % within each LCA_Class
threshold = 0.05
ct_pct_display = ct_pct.where(ct_pct >= threshold, np.nan)

fig = px.imshow(ct_pct_display, text_auto=".0%", aspect="auto", color_continuous_scale="Blues")
fig.update_layout(title=" k_proto_labs_25 distribution within each hdb_labels_281_50 ", coloraxis_showscale = False)
fig.show()